# Basic Python Logging

Goal: create a basic logger that logs messages to the console — understand why `logging` replaces `print`, how log levels work, and how to configure a logger with a handler and formatter.

## 1. Why not just `print`?

**Why we need it:** `print` has no notion of severity, no timestamp, no source location, and no way to turn messages off without deleting code. The `logging` module gives every message a *level* (how serious it is), and lets you filter, format, and redirect messages (console, file, network) without touching the call sites.

In [1]:
import logging

# without any configuration, the root logger only shows WARNING and above
logging.debug("this won't print — below default level")
logging.info("neither will this")
logging.warning("this will print — WARNING is the default threshold")
logging.error("this will print too")

ERROR:root:this will print too


## 2. Log levels

**Why we need it:** each level maps to a severity number — filtering is just "show me everything at or above this number." Knowing the order tells you what a given threshold will and won't show.

| Level      | Numeric value | Typical use                                  |
|------------|----------------|-----------------------------------------------|
| `DEBUG`    | 10             | detailed diagnostic info, dev-only            |
| `INFO`     | 20             | confirmation things are working as expected   |
| `WARNING`  | 30             | something unexpected, but still working       |
| `ERROR`    | 40             | a real problem — some functionality failed    |
| `CRITICAL` | 50             | the program may not be able to continue       |

## 3. `basicConfig` — the quick way to a console logger

**Why we need it:** `logging.basicConfig()` is a one-call shortcut that attaches a `StreamHandler` (writes to the console) to the root logger, sets its level, and applies a message format. It only has an effect the *first* time it's called (unless `force=True`), because its job is to set up the root logger once at program start.

In [2]:
# force=True re-applies config even though logging calls above already touched the root logger —
# only needed here because we're re-running basicConfig in the same notebook session
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)

logging.debug("now visible — level is DEBUG")
logging.info("application started")
logging.warning("disk usage is high")
logging.error("failed to save file")
logging.critical("out of memory")

12:25:57 | DEBUG    | root | now visible — level is DEBUG


12:25:57 | INFO     | root | application started


12:25:57 | WARNING  | root | disk usage is high


12:25:57 | ERROR    | root | failed to save file


12:25:57 | CRITICAL | root | out of memory


## 4. Named loggers — `getLogger(__name__)`

**Why we need it:** using the root logger directly (`logging.info(...)`) works for a one-off script, but in a real module you want to know *which part* of the app produced a message. `getLogger(__name__)` gives each module its own logger named after its module path, all feeding into the same root configuration — that's why `%(name)s` showed up in the format above.

In [3]:
logger = logging.getLogger(__name__)  # e.g. "__main__" in a notebook, "payments.orders" in a real module

logger.info("processing order #1042")
logger.warning("order #1042 has no shipping address")

12:25:57 | INFO     | __main__ | processing order #1042


12:25:57 | WARNING  | __main__ | order #1042 has no shipping address


## 5. Building a console logger by hand

**Why we need it:** `basicConfig` configures the *root* logger, which is fine for scripts but risky for reusable modules — importing your module would silently change every consumer's root logging setup. The explicit pattern below builds a logger, a handler, and a formatter separately and wires them together, so it's self-contained and safe to reuse.

The pieces:
- **Logger** — the entry point you call `.debug()` / `.info()` / etc. on. Its level is the first filter.
- **Handler** — decides *where* a message goes (console, file, network...). `StreamHandler` writes to the console.
- **Formatter** — decides *what the message looks like* once it reaches a handler.

In [4]:
def build_console_logger(name: str, level: int = logging.DEBUG) -> logging.Logger:
    logger = logging.getLogger(name)
    logger.setLevel(level)
    logger.propagate = False  # don't also send records up to the root logger's handlers

    if not logger.handlers:  # avoid attaching duplicate handlers if this is called more than once
        handler = logging.StreamHandler()  # defaults to sys.stderr
        handler.setLevel(level)

        formatter = logging.Formatter(
            fmt="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
            datefmt="%H:%M:%S",
        )
        handler.setFormatter(formatter)

        logger.addHandler(handler)

    return logger


app_logger = build_console_logger("checkout_service")

app_logger.debug("validating cart contents")
app_logger.info("payment authorized for order #1042")
app_logger.error("inventory service timed out")

12:25:57 | DEBUG    | checkout_service | validating cart contents


12:25:57 | INFO     | checkout_service | payment authorized for order #1042


12:25:57 | ERROR    | checkout_service | inventory service timed out


## 6. Logging exceptions with a traceback

**Why we need it:** `logger.error("it broke")` inside an `except` block records that something failed but throws away *why*. `logger.exception(...)` — only valid inside an `except` block — logs at `ERROR` level and automatically attaches the full traceback.

In [5]:
try:
    1 / 0
except ZeroDivisionError:
    app_logger.exception("failed to compute order total")

12:25:57 | ERROR    | checkout_service | failed to compute order total
Traceback (most recent call last):
  File "/tmp/ipykernel_65466/3557615082.py", line 2, in <module>
    1 / 0
    ~~^~~
ZeroDivisionError: division by zero


## Summary

- `print` has no severity, filtering, or routing — `logging` does.
- Levels (`DEBUG` < `INFO` < `WARNING` < `ERROR` < `CRITICAL`) let you filter by "how serious."
- `logging.basicConfig(...)` is the fast path: configures the root logger's handler, level, and format in one call.
- `logging.getLogger(__name__)` gives each module its own named logger.
- For reusable modules, build the logger/handler/formatter explicitly instead of relying on the shared root logger via `basicConfig`.
- `logger.exception(...)` inside an `except` block logs the error *with* its traceback.